# YOLO11 학습 — 장난감 회수 로봇 (bag / cube / doll / duck / robot)

`one_stage_node`의 정찰 탐지 모델(`config.YOLO_MODEL_PATH`)을 만들기 위한 학습 노트북.

**진행 순서**
1. 이 저장소의 `training/prepare_dataset.py`를 로컬에서 먼저 실행해 만든 `dataset.zip`을 업로드
2. ultralytics 설치 후 YOLO11n으로 학습
3. 학습된 `best.pt`를 다운로드 -> `~/cobot_ws/src/yolo_detect/resource/yolo_toy.pt`로 배치

런타임: **GPU**로 설정할 것 (런타임 -> 런타임 유형 변경 -> T4 GPU)

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 1. 데이터셋 업로드

로컬에서 `python3 training/prepare_dataset.py` 실행 후 생성되는
`training/dataset.zip`(약 22MB, 500장 + data.yaml)을 아래 셀 실행 후 뜨는
업로드 창에서 선택한다.

In [ ]:
from google.colab import files

uploaded = files.upload()  # dataset.zip 선택
assert "dataset.zip" in uploaded, "dataset.zip 파일을 업로드해야 합니다"

In [ ]:
import os
import zipfile

import yaml

with zipfile.ZipFile("dataset.zip", "r") as zf:
    zf.extractall(".")

# data.yaml의 path(".")를 절대경로로 다시 씀 — ultralytics 버전에 따라 상대경로가
# yaml 파일 위치가 아니라 현재 작업 디렉터리(cwd) 기준으로 해석되면서
# "missing path '/content/images/val'"처럼 엉뚱한 곳을 찾는 문제가 있어 이렇게 고정한다.
data_yaml_path = "dataset/data.yaml"
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)
data_cfg["path"] = os.path.abspath("dataset")
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, allow_unicode=True)

!echo "train:" $(ls dataset/images/train | wc -l) "장"
!echo "val:" $(ls dataset/images/val | wc -l) "장"
!cat dataset/data.yaml

## 2. 학습

500장(클래스당 100장)으로는 데이터가 많지 않으므로 사전학습 가중치(`yolo11n.pt`)에서
fine-tuning하는 것이 필수. `imgsz=640`은 원본 해상도(640x480)와 맞춘 값.

**데이터 증강(2026-07-22 추가)**: 실기 정찰 테스트에서 bbox 시각화로 직접 확인한
실패 사례들(배경 창문을 cube로, 초록 테이프를 duck으로, bag을 duck으로 오분류)이
근거. 학습 이미지(`prepare_dataset.py`가 원본 그대로 사용, 물체 1개·비교적 정면
구도)와 실제 정찰 촬영(웨이포인트마다 크게 기운 각도로 내려다봄, 300~900mm 거리
편차, 배경에 창문/선반/테이프 등 잡다한 물체 포함) 사이 괴리가 커서, 학습 시점에
각도·거리·배경 다양성을 인위적으로 늘려야 이 괴리가 줄어들 것으로 판단. 아래
`model.train()`의 증강 인자들이 그 대응:
- `degrees=25`, `perspective=0.0008`, `shear=8.0`: 정찰 웨이포인트들의 기운 시야각(rx
  66~127°) 대응 — 기본값(0)은 회전/원근 왜곡을 전혀 안 줌
- `scale=0.6`: 물체가 300~900mm 거리에서 다양한 크기로 잡히는 것 대응(기본 0.5보다 약간 넓힘)
- `translate=0.2`: bbox가 프레임 중앙에만 있지 않도록(기본 0.1보다 넓힘)
- `hsv_h/hsv_s/hsv_v`: 조명 편차 대응, 기본값보다 약간 상향
- `mosaic=1.0`(기본값 유지): 4장을 합성해 "물체 1개짜리" 원본 데이터를 다물체 장면처럼
  보이게 함 — 지금 데이터셋의 가장 큰 약점(이미지당 물체 1개)을 부분적으로 완화
- `mixup=0.1`: 데이터가 적어 과적합 위험이 있어 약하게 추가
- `erasing=0.4`(기본값 유지): 부분 가림(occlusion) 대응
- `copy_paste=0.0`: VOC bbox만 있고 세그멘테이션 마스크가 없어 비활성 — 마스크 없이 켜면
  효과가 제한적이라 끔(마스크 있는 데이터로 바뀌면 재검토)

이 값들은 전부 근본적인 데이터 부족(클래스당 100장)을 대체하진 못함 — 실기에서 성능이
여전히 부족하면 클래스당 이미지 수를 늘리거나(특히 다양한 각도/배경/거리로), 실제
정찰 웨이포인트 구도와 비슷한 사진을 직접 추가하는 게 증강보다 효과가 클 수 있음.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # nano — 데이터 적고, one_stage_node가 정찰 종료 후 배치 처리라 추론 속도보다 정확도/용량 균형 우선

results = model.train(
    data="dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,       # 20 epoch 동안 val 성능 개선 없으면 조기 종료
    name="toy_retrieval_yolo11n",
    # project는 일부러 지정 안 함 — ultralytics 버전마다 project/name을 실제
    # 폴더 경로로 조합하는 방식이 달라(예: runs/detect/<project>/<name> 식으로
    # 중첩되기도 함), 기본값에 맡기고 아래 5번에서 model.trainer.best로
    # 실제 저장 경로를 직접 물어봐서 하드코딩을 피한다.

    # ---- 데이터 증강 (2026-07-22, 위 마크다운 설명 참고) ----
    degrees=25.0,      # 회전 — 정찰 웨이포인트의 기운 시야각 대응(기본 0)
    perspective=0.0008,  # 원근 왜곡 — 위와 같은 이유(기본 0)
    shear=8.0,         # 전단 왜곡(기본 0)
    scale=0.6,         # 크기 변화폭 — 300~900mm 거리 편차 대응(기본 0.5)
    translate=0.2,     # 위치 이동폭(기본 0.1)
    hsv_h=0.02,        # 색상 jitter(기본 0.015)
    hsv_s=0.8,         # 채도 jitter(기본 0.7)
    hsv_v=0.5,         # 명도 jitter — 조명 편차 대응(기본 0.4)
    mixup=0.1,         # 이미지 합성 — 데이터 적어 과적합 방지용 약하게(기본 0.0)
    copy_paste=0.0,    # 세그멘테이션 마스크 없어 비활성(명시)
    # mosaic=1.0, fliplr=0.5, erasing=0.4는 기본값 그대로 사용(명시 안 함) —
    # mosaic이 "이미지당 물체 1개" 데이터의 가장 큰 약점을 완화해줌
)

## 3. 검증 (val set 기준 지표 확인)

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 4. 샘플 예측 확인 (선택)

In [ ]:
import glob

sample = glob.glob("dataset/images/val/*.jpg")[:5]
pred_results = model.predict(sample, save=True, conf=0.4)
for r in pred_results:
    print(r.path, "->", [(model.names[int(c)], float(conf)) for c, conf in zip(r.boxes.cls, r.boxes.conf)])

## 5. 결과 다운로드

`best.pt`를 `yolo_toy.pt`로 이름을 바꿔 다운로드한다. 다운로드 후
`~/cobot_ws/src/yolo_detect/resource/yolo_toy.pt`에 넣으면
`config.YOLO_MODEL_PATH`가 바로 이 경로를 가리키므로 `one_stage_node`가 그대로 사용한다.

In [ ]:
import glob
import shutil

from google.colab import files

try:
    best_path = str(model.trainer.best)  # 방금 학습에 쓴 model 객체가 기억하는 실제 저장 경로
except (NameError, AttributeError):
    # 커널을 재시작해서 model 객체가 없거나(런타임 끊김 등) trainer.best를 못 구할 때의 대비책 —
    # runs 폴더 전체에서 best.pt를 찾는다(방금 학습한 게 1개뿐이라고 가정).
    candidates = glob.glob("**/toy_retrieval_yolo11n*/weights/best.pt", recursive=True)
    assert candidates, "best.pt를 못 찾음 — 학습이 끝났는지, runs 폴더가 남아있는지 확인 필요"
    best_path = candidates[0]

print("best.pt 위치:", best_path)
shutil.copy(best_path, "yolo_toy.pt")
files.download("yolo_toy.pt")